# ML-05b (C#) : Séries temporelles classiques — stationnarité, ACF/PACF, ARIMA/SARIMA, lissage exponentiel

> **Socle statistique.** Ce notebook pose les fondations que [ML-05-TimeSeries-CSharp](ML-05-TimeSeries-CSharp.ipynb) (SSA, ML.NET) applique en boîte noire et que [ML-05-TimeSeries-Python](ML-05-TimeSeries-Python.ipynb) (STL + SARIMA) applique mécaniquement : **tester la stationnarité**, **lire une ACF/PACF**, **ajuster un ARIMA**, **comprendre pourquoi différencier**, et **comparer honnêtement** ARIMA, lissage exponentiel et baseline naïve.



## Objectifs d'apprentissage

À l'issue de ce notebook, vous saurez :

1. **Décomposer** une série en tendance + saisonnalité + résidu (`seasonal_decompose`), et lire chaque composante.
2. **Tester la stationnarité** avec ADF (H0, p-value, conclusion écrite) et **différencier** pour la rétablir.
3. **Lire une ACF/PACF** pour identifier les ordres `p` et `q` (les barres hors bande de confiance), y compris sur un AR(2) synthétique.
4. **Ajuster un SARIMA** depuis la lecture ACF/PACF, **diagnostiquer** les résidus (ACF plate + Ljung-Box) et **prévoir** avec intervalle de confiance.
5. **Lisser** par simple / Holt / Holt-Winters, et savoir quand le lissage égale ARIMA à moindre coût intellectuel.
6. **Comparer honnêtement** (ARIMA vs Holt-Winters vs naïf) sur un backtest glissant — la baseline naïve qui gagne parfois **est** la leçon.

**Relation aux autres notebooks** : ML-05 (SSA) = le même problème en ML.NET ; ML-05-TimeSeries-Python = l'application STL+SARIMA ; ce notebook = le **pourquoi** statistique que les deux supposent. Du côté quant, les notebooks HAR de QuantConnect poussent la même famille vers les features modernes (voir la conclusion).

**Navigation** : [Version Python](ML-05b-Series-Temporelles-Classiques-Python.ipynb) · [Index](README.md).

**Prérequis** : statistiques élémentaires, tableaux, régression ; **durée indicative** : 90 minutes.


**Politique de portage** : statsmodels conserve les vraies décompositions, tests, SARIMA/ARIMA et lissages via Python.NET. SSA de ML.NET est un autre modèle et ne fournit pas cette leçon exacte. Les données, grilles, métriques, backtests et figures sont orchestrés en C#. Les tirages NumPy servent à conserver les mêmes innovations synthétiques. Aucun script Python ne rejoue le carnet entier.


In [1]:
#r "nuget: pythonnet,3.1.0"
#load "../../Config/PythonNetLoader.cs"
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Text.Json;
using System.Security.Cryptography;
using Python.Runtime;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
if(!PythonEngine.IsInitialized) {
    PythonNetLoader.Initialize(typeof(PythonEngine).Assembly,"statsmodels");
    PythonEngine.BeginAllowThreads();
}
PyObject JsonPy<T>(T value) { dynamic json=Py.Import("json"); return json.loads(JsonSerializer.Serialize(value)); }
PyObject ArrayPy(double[] values) { using(Py.GIL()){dynamic np=Py.Import("numpy");using PyObject parsed=JsonPy(values);return np.array(parsed,dtype:np.float64);} }
double[] Vector(PyObject values) { using(Py.GIL()){using PyObject list=((dynamic)values).tolist();return list.As<double[]>();} }
double[][] Matrix(PyObject values) { using(Py.GIL()){using PyObject list=((dynamic)values).tolist();return list.As<double[][]>();} }
(double Statistic,double PValue,int Lags,int N) Adf(double[] values) {
    using(Py.GIL()){dynamic st=Py.Import("statsmodels.tsa.stattools");using PyObject a=ArrayPy(values);using PyObject r=st.adfuller(a);
    return(r.GetItem(0).As<double>(),r.GetItem(1).As<double>(),r.GetItem(2).As<int>(),r.GetItem(3).As<int>());}
}
(double[] Acf,double[] Pacf) Correlations(double[] values,int lags) {
    using(Py.GIL()){dynamic st=Py.Import("statsmodels.tsa.stattools");using PyObject a=ArrayPy(values);using PyObject x=st.acf(a,nlags:lags);using PyObject p=st.pacf(a,nlags:lags,method:"ywm");return(Vector(x),Vector(p));}
}
PyObject FitSarima(double[] values,int[] order,int[] seasonal) {
    using(Py.GIL()){dynamic sm=Py.Import("statsmodels.tsa.statespace.sarimax");using PyObject a=ArrayPy(values);using PyObject o=JsonPy(order);using PyObject s=JsonPy(seasonal);using PyObject model=sm.SARIMAX(a,order:o,seasonal_order:s);return ((dynamic)model).fit(disp:false);}
}
PyObject FitSmoothing(double[] values,string kind) {
    using(Py.GIL()){dynamic sm=Py.Import("statsmodels.tsa.holtwinters");using PyObject a=ArrayPy(values);using PyObject model=kind=="HW"?sm.ExponentialSmoothing(a,trend:"add",seasonal:"add",seasonal_periods:7):kind=="Holt"?sm.Holt(a):sm.SimpleExpSmoothing(a);return ((dynamic)model).fit();}
}
double Aic(PyObject model) {using(Py.GIL()){return ((PyObject)((dynamic)model).aic).As<double>();}}
double[] Forecast(PyObject model,int steps) {using(Py.GIL()){using PyObject f=((dynamic)model).forecast(steps);return Vector(f);}}
(double[] Mean,double[][] Interval) ForecastInterval(PyObject model,int steps) {
    using(Py.GIL()){using PyObject f=((dynamic)model).get_forecast(steps);using PyObject mean=((dynamic)f).predicted_mean;using PyObject ci=((dynamic)f).conf_int(alpha:.05);return(Vector(mean),Matrix(ci));}
}
double[][] Decompose(double[] values,int period) {
    using(Py.GIL()){dynamic sm=Py.Import("statsmodels.tsa.seasonal");using PyObject a=ArrayPy(values);using PyObject d=sm.seasonal_decompose(a,model:"additive",period:period);return new[]{"observed","trend","seasonal","resid"}.Select(key=>{using PyObject v=d.GetAttr(key);return Vector(v);}).ToArray();}
}
double[] Difference(double[] values,int lag=1) => Enumerable.Range(lag,values.Length-lag).Select(i=>values[i]-values[i-lag]).ToArray();
double Mape(double[] truth,double[] prediction) {
    if(truth.Length==0||truth.Length!=prediction.Length||truth.Any(x=>!double.IsFinite(x)||x==0)||prediction.Any(x=>!double.IsFinite(x))) throw new ArgumentException("MAPE : dimensions, valeurs finies et dénominateurs non nuls requis.");
    return truth.Zip(prediction,(x,y)=>Math.Abs((x-y)/x)).Average()*100;
}
double MeanFinite(double[] a)=>a.Where(double.IsFinite).Average();
double StdFinite(double[] a){var b=a.Where(double.IsFinite).ToArray();double m=b.Average();return Math.Sqrt(b.Sum(x=>(x-m)*(x-m))/(b.Length-1));}
using(Py.GIL()){dynamic sys=Py.Import("sys"),sm=Py.Import("statsmodels"),np=Py.Import("numpy"),sp=Py.Import("scipy");Console.WriteLine($".NET {Environment.Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | CPython {sys.version.split()[0]} | statsmodels {sm.__version__} | NumPy {np.__version__} | SciPy {sp.__version__}");}
Console.WriteLine("Statsmodels réel : décomposition, ADF/ACF/PACF, SARIMA/ARIMA et trois lissages ; orchestration et métriques C# prêtes.");
(double[] Values,double[][] CenteredIntervals) CorrelationInterval(double[] values,int lags,bool partial=false) {
    using(Py.GIL()){dynamic st=Py.Import("statsmodels.tsa.stattools");using PyObject a=ArrayPy(values);using PyObject r=partial?st.pacf(a,nlags:lags,method:"ywm",alpha:.05):st.acf(a,nlags:lags,alpha:.05);using PyObject x=r.GetItem(0);using PyObject c=r.GetItem(1);var v=Vector(x);var ci=Matrix(c);for(int i=0;i<ci.Length;i++)for(int j=0;j<2;j++)ci[i][j]-=v[i];return(v,ci);}
}


Installed Packages pythonnet, 3.1.0

.NET 10.0.14 | Python.NET 3.1.0.0 | CPython 3.13.16 | statsmodels 0.14.6 | NumPy 2.4.4 | SciPy 1.17.0


Statsmodels réel : décomposition, ADF/ACF/PACF, SARIMA/ARIMA et trois lissages ; orchestration et métriques C# prêtes.


### Figures statiques du parcours

Les six expériences graphiques du source sont conservées sous forme SVG, avec panneaux séparés, axes, unités, légendes et zones de confiance. Le rendu ne dépend ni de JavaScript ni d’un CDN.


In [2]:
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((value,writer)=>writer.Write(value.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
string Esc(string s)=>System.Security.SecurityElement.Escape(s)??"";
string Plot(string title,(string Name,double[] Y,string Color)[] series,string xLabel="indice quotidien",string yLabel="ventes",double[][] interval=null,int intervalOffset=0,double? band=null) {
    int length=series.Max(s=>s.Y.Length);var finite=series.SelectMany(s=>s.Y).Where(double.IsFinite).ToList();
    if(interval!=null)finite.AddRange(interval.SelectMany(x=>x).Where(double.IsFinite));if(band.HasValue){finite.Add(band.Value);finite.Add(-band.Value);}
    double min=finite.Min(),max=finite.Max();if(max<=min)max=min+1;double pad=(max-min)*.08;min-=pad;max+=pad;
    double X(int i)=>65+720.0*i/Math.Max(1,length-1);double Y(double v)=>285-225*(v-min)/(max-min);
    var b=new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 820 365' role='img' aria-label='{Esc(title)}'><rect width='820' height='365' fill='white'/><text x='410' y='24' text-anchor='middle' font-family='sans-serif' font-size='16'>{Esc(title)}</text>");
    for(int k=0;k<=4;k++){double v=min+(max-min)*k/4.0,y=Y(v);b.Append($"<path d='M65 {F(y)}H785' stroke='#ddd'/><text x='59' y='{F(y+4)}' text-anchor='end' font-size='11'>{F(v)}</text>");}
    for(int k=0;k<=5;k++){int i=(int)Math.Round((length-1)*k/5.0);double x=X(i);b.Append($"<path d='M{F(x)} 285v5' stroke='#333'/><text x='{F(x)}' y='307' text-anchor='middle' font-size='11'>{i}</text>");}
    b.Append($"<path d='M65 60V285H785' fill='none' stroke='#333'/><text x='420' y='327' text-anchor='middle' font-size='12'>{Esc(xLabel)}</text><text x='16' y='173' transform='rotate(-90 16 173)' text-anchor='middle' font-size='12'>{Esc(yLabel)}</text>");
    if(band.HasValue)foreach(double v in new[]{-band.Value,band.Value})b.Append($"<path d='M65 {F(Y(v))}H785' stroke='#789' stroke-dasharray='4 3'/>");
    if(interval!=null){var points=interval.Select((a,i)=>(x:X(i+intervalOffset),y:Y(a[0]))).Concat(interval.Select((a,i)=>(x:X(i+intervalOffset),y:Y(a[1]))).Reverse().Take(interval.Length));b.Append("<polygon fill='#e2e9f4' points='"+string.Join(" ",points.Select(p=>$"{F(p.x)},{F(p.y)}"))+"'/>");}
    int legendX=70;
    foreach(var s in series){var path=new StringBuilder();bool active=false;for(int i=0;i<s.Y.Length;i++){double v=s.Y[i];if(!double.IsFinite(v)){active=false;continue;}path.Append($"{(active?"L":"M")}{F(X(i))} {F(Y(v))}");active=true;}b.Append($"<path d='{path}' fill='none' stroke='{s.Color}' stroke-width='1.5'/><path d='M{legendX} 347h18' stroke='{s.Color}' stroke-width='2'/><text x='{legendX+23}' y='351' font-size='11'>{Esc(s.Name)}</text>");legendX+=Math.Min(220,38+s.Name.Length*6);}
    return b.Append("</svg>").ToString();
}
string Stack(string[] charts){return "<div style='background:white'>"+string.Join("",charts)+"</div>";}
void Show(string title,(string Name,double[] Y,string Color)[] series,string xLabel="indice quotidien",string yLabel="ventes",double[][] interval=null,int intervalOffset=0,double? band=null)=>display(new PlotSvg(Plot(title,series,xLabel,yLabel,interval,intervalOffset,band)));
Console.WriteLine("Rendu SVG statique : fonds blancs, axes et unités, légendes et intervalle ; aucune ressource externe.");


Rendu SVG statique : fonds blancs, axes et unités, légendes et intervalle ; aucune ressource externe.


## Partie 1 : Décomposition — tendance + saisonnalité + résidu

On commence par **voir** ce qui compose une série temporelle. Le modèle additif classique est :

```
série(t) = tendance(t) + saisonnalité(t) + résidu(t)
```

`statsmodels.seasonal_decompose` sépare ces trois composantes. On le démontre d'abord sur une série **synthétique franche** (tendance + saisonnalité 12 + bruit), pour que les trois composantes soient lisibles, puis sur la **vraie série du dépôt** (`daily-sales.csv`, celle que ML-05 utilise).

### Exemple guidé — série synthétique franche

La cellule ci-dessous fabrique la série : une tendance linéaire de pente 0,4, une saisonnalité sinusoïdale de période 12, et un bruit gaussien.

In [3]:
double[] noise;
using(Py.GIL()){dynamic np=Py.Import("numpy");using PyObject z=np.random.default_rng(7).normal(0.0,4.0,120);noise=Vector(z);}
var synth=Enumerable.Range(0,120).Select(t=>50+.4*t+15*Math.Sin(2*Math.PI*t/12)+noise[t]).ToArray();
var synthParts=Decompose(synth,12);
var partNames=new[]{"observée","tendance","saisonnière","résidu"};
display(new PlotSvg(Stack(synthParts.Select((v,i)=>Plot("Série synthétique — "+partNames[i],new[]{(partNames[i],v,"#2677aa")},"indice mensuel","unités")).ToArray())));
Console.WriteLine($"120 mois, période 12, pente 0,4/mois ; tendance moyenne {MeanFinite(synthParts[1]):F1}, amplitude saisonnière {synthParts[2].Max()-synthParts[2].Min():F1}, résidu moyenne {MeanFinite(synthParts[3]):F3}, écart-type {StdFinite(synthParts[3]):F2}");


Série synthétique — observée 30.089 51.944 73.799 95.654 117.509 0 24 48 71 95 119 indice mensuel unités <path d='M65 233.741L71.05 210.345L77.101 201.082L83.151 201.227L89.202 200.882L95.252 219.512L101.303 226.957L107.353 232.053L113.403 264.019L119.454 269.483L125.504 251.85L131.555 238.058L137.605 220.314L143.655 210.646L149.706 186.207L155.756 172.545L161.807 197.686L167.857 201.659L173.908 234.796L179.958 246.772L186.008 265.559L192.059 253.161L198.109 257.587L204.16 226.585L210.21 207.431L216.261 190.637L222.311 199.462L228.361 172.895L234.412 171.992L240.462 183.427L246.513 218.621L252.563 226.06L258.613 244.318L264.664 246.714L270.714 221.262L276.765 225.338L282.815 197.026L288.866 167.253L294.916 167.205L300.966 156.145L307.017 158.001L313.067 171.583L319.118 203.126L325.168 208.004L331.218 207.9L337.269 241.96L343.319 210.983L349.37 203.441L355.42 190.941L361.471 143.409L367.521 140.995L373.571 154.988L379.622 146.017L385.672 153.948L391.723 180.103L397.773 189.403L403.824 210.22L409.874 206.808L415.924 192.666L421.975 199.271L428.025 169.891L434.076 156.419L440.126 135.178L446.176 142.509L452.227 140.394L458.277 149.551L464.328 156.552L470.378 172.289L476.429 210.807L482.479 209.505L488.529 188.462L494.58 200.473L500.63 164.397L506.681 140.297L512.731 111.193L518.782 110.835L524.832 125.444L530.882 138.972L536.933 156.027L542.983 156.04L549.034 189.233L555.084 192.096L561.134 179.138L567.185 168.85L573.235 149.305L579.286 138.41L585.336 111.899L591.387 110.145L597.437 97.716L603.487 116.099L609.538 141.345L615.588 152.49L621.639 175.971L627.689 165.783L633.739 170.469L639.79 149.247L645.84 148.21L651.891 111.017L657.941 116.806L663.992 114.178L670.042 100.502L676.092 119.733L682.143 127.054L688.193 123.906L694.244 168.681L700.294 170.685L706.345 155.944L712.395 137.823L718.445 124.381L724.496 104.352L730.546 79.84L736.597 75.517L742.647 95.655L748.697 98.929L754.748 116.025L760.798 145.518L766.849 145.089L772.899 160.74L778.95 135.697L785 128.56' fill='none' stroke='#2677aa' stroke-width='1.5'/> observée Série synthétique — tendance 48.586 61.052 73.517 85.982 98.447 0 24 48 71 95 119 indice mensuel unités <path d='M101.303 269.483L107.353 268.524L113.403 267.459L119.454 264.277L125.504 261.948L131.555 260.411L137.605 259.679L143.655 261.327L149.706 262.515L155.756 261.435L161.807 260.662L167.857 260.243L173.908 258.464L179.958 256.061L186.008 255.567L192.059 256.561L198.109 254.71L204.16 251.501L210.21 248.987L216.261 246.293L222.311 243.228L228.361 241.205L234.412 238.08L240.462 235.336L246.513 234.484L252.563 232.016L258.613 227.951L264.664 224.371L270.714 222.125L276.765 220.238L282.815 218.241L288.866 215.79L294.916 211.81L300.966 208.802L307.017 207.704L313.067 205.354L319.118 203.309L325.168 201.123L331.218 197.466L337.269 195.467L343.319 194.507L349.37 192.343L355.42 189.373L361.471 186.332L367.521 185.143L373.571 182.744L379.622 178.838L385.672 177.196L391.723 175.353L397.773 174.766L403.824 175.292L409.874 173.955L415.924 172.633L421.975 171.901L428.025 169.859L434.076 166.888L440.126 165.681L446.176 165.921L452.227 165.811L458.277 165.591L464.328 165.278L470.378 163.698L476.429 160.768L482.479 156.702L488.529 153.296L494.58 151.431L500.63 150.62L506.681 149.395L512.731 146.632L518.782 143.784L524.832 141.831L530.882 138.84L536.933 135.427L542.983 134.187L549.034 134.1L555.084 134.101L561.134 132.025L567.185 128.329L573.235 125.585L579.286 124.253L585.336 123.025L591.387 120.134L597.437 117.579L603.487 115.513L609.538 114.001L615.588 111.92L621.639 110.277L627.689 110.931L633.739 111.429L639.79 111.898L645.84 111.119L651.891 107.987L657.941 105.366L663.992 105.192L670.042 104.489L676.092 102.593L682.143 100.018L688.193 97.79L694.244 94.603L700.294 89.078L706.345 85.9L712.395 84.026L718.445 81.7L724.496 82.473L730.546 82.329L736.597 79.879L742.647 77.673L748.697 75.517' fill='none' stroke='#2677aa' stroke-width='1.5'/> tendance Série synthétique — saisonnière -18.283 -9.626 -0.969 7.688 16.345 0 24 48 71 9

120 mois, période 12, pente 0,4/mois ; tendance moyenne 73.2, amplitude saisonnière 29.9, résidu moyenne 0.053, écart-type 3.07


### Interprétation de la décomposition synthétique

La tendance croît de **0,4 par mois** dans le générateur. La décomposition donne une tendance moyenne voisine de 73 et une amplitude saisonnière proche de 30 ; les valeurs exactes sont imprimées. La composante saisonnière est périodique par construction de l'algorithme : cela ne prouve pas une périodicité parfaite d'une série observée. Le résidu, de moyenne proche de zéro et d'écart-type proche de 3, reste de l'ordre du bruit injecté (écart-type 4), dont la décomposition absorbe une partie. Une moyenne et une variance empiriques ne suffisent pas à établir la stationnarité ; celle du bruit est ici une hypothèse du générateur.

La prévision cherche à exploiter tendance et saisonnalité sans confondre le bruit du tirage avec un signal prévisible.


### Application à la série fournie dans le dépôt — daily-sales.csv

On utilise exactement le fichier de ventes quotidiennes partagé avec ML-05 : période 2023–2024, cycle hebdomadaire (période 7). Le fichier est une donnée pédagogique du dépôt ; son utilisation ne constitue pas une affirmation sur une provenance commerciale indépendante. La décomposition est calculée sur l'entraînement 2023 seulement.


In [4]:
var csv=File.ReadAllLines("daily-sales.csv");
var records=csv.Skip(1).Where(x=>!string.IsNullOrWhiteSpace(x)).Select(line=>{var fields=line.Split(',');return(Date:DateTime.Parse(fields[0],CultureInfo.InvariantCulture),Sales:double.Parse(fields[1],CultureInfo.InvariantCulture));}).OrderBy(x=>x.Date).ToArray();
if(records.Any(x=>!double.IsFinite(x.Sales))||records.Zip(records.Skip(1),(a,b)=>(b.Date-a.Date).Days).Any(d=>d!=1))throw new Exception("Ventes : données finies et jours consécutifs requis.");
var sales=records.Select(x=>x.Sales).ToArray();var train=records.Where(x=>x.Date.Year==2023).Select(x=>x.Sales).ToArray();var test=records.Where(x=>x.Date.Year==2024).Select(x=>x.Sales).ToArray();
Console.WriteLine($"daily-sales.csv SHA256 {Convert.ToHexString(SHA256.HashData(File.ReadAllBytes("daily-sales.csv"))).ToLowerInvariant()}");
Console.WriteLine($"{records[0].Date:yyyy-MM-dd} → {records[^1].Date:yyyy-MM-dd} : {sales.Length} jours ; train2023 {train.Length}, moyenne {train.Average():F1} ; test2024 {test.Length}, moyenne {test.Average():F1} ; croissance {(test.Average()/train.Average()-1)*100:F1} %");
var trainParts=Decompose(train,7);
display(new PlotSvg(Stack(trainParts.Select((v,i)=>Plot("Ventes 2023 — "+partNames[i],new[]{(partNames[i],v,"#2677aa")},"jour depuis le 01/01/2023","ventes")).ToArray())));
Console.WriteLine($"Tendance 2023 aux extrémités disponibles : {trainParts[1].First(double.IsFinite):F1} → {trainParts[1].Last(double.IsFinite):F1}");


daily-sales.csv SHA256 072b30f55ac726f7bfe31cfb78fd5e085669d1bc992d0d5459db7e09d49ae049


2023-01-01 → 2024-12-31 : 731 jours ; train2023 365, moyenne 2023.0 ; test2024 366, moyenne 2024.0 ; croissance 0.0 %


Ventes 2023 — observée 2022.92 2023.21 2023.5 2023.79 2024.08 0 73 146 218 291 364 jour depuis le 01/01/2023 ventes <path d='M65 269.483L66.978 269.483L68.956 269.483L70.934 269.483L72.912 269.483L74.89 269.483L76.868 269.483L78.846 269.483L80.824 269.483L82.802 269.483L84.78 269.483L86.758 269.483L88.736 269.483L90.714 269.483L92.692 269.483L94.67 269.483L96.648 269.483L98.626 269.483L100.604 269.483L102.582 269.483L104.56 269.483L106.538 269.483L108.516 269.483L110.495 269.483L112.473 269.483L114.451 269.483L116.429 269.483L118.407 269.483L120.385 269.483L122.363 269.483L124.341 269.483L126.319 269.483L128.297 269.483L130.275 269.483L132.253 269.483L134.231 269.483L136.209 269.483L138.187 269.483L140.165 269.483L142.143 269.483L144.121 269.483L146.099 269.483L148.077 269.483L150.055 269.483L152.033 269.483L154.011 269.483L155.989 269.483L157.967 269.483L159.945 269.483L161.923 269.483L163.901 269.483L165.879 269.483L167.857 269.483L169.835 269.483L171.813 269.483L173.791 269.483L175.769 269.483L177.747 269.483L179.725 269.483L181.703 269.483L183.681 269.483L185.659 269.483L187.637 269.483L189.615 269.483L191.593 269.483L193.571 269.483L195.549 269.483L197.527 269.483L199.505 269.483L201.484 269.483L203.462 269.483L205.44 269.483L207.418 269.483L209.396 269.483L211.374 269.483L213.352 269.483L215.33 269.483L217.308 269.483L219.286 269.483L221.264 269.483L223.242 269.483L225.22 269.483L227.198 269.483L229.176 269.483L231.154 269.483L233.132 269.483L235.11 269.483L237.088 269.483L239.066 269.483L241.044 269.483L243.022 269.483L245 269.483L246.978 269.483L248.956 269.483L250.934 269.483L252.912 269.483L254.89 269.483L256.868 269.483L258.846 269.483L260.824 269.483L262.802 269.483L264.78 269.483L266.758 269.483L268.736 269.483L270.714 269.483L272.692 269.483L274.67 269.483L276.648 269.483L278.626 269.483L280.604 269.483L282.582 269.483L284.56 269.483L286.538 269.483L288.516 269.483L290.495 269.483L292.473 269.483L294.451 269.483L296.429 269.483L298.407 269.483L300.385 269.483L302.363 269.483L304.341 269.483L306.319 269.483L308.297 269.483L310.275 269.483L312.253 269.483L314.231 269.483L316.209 269.483L318.187 269.483L320.165 269.483L322.143 269.483L324.121 269.483L326.099 269.483L328.077 269.483L330.055 269.483L332.033 269.483L334.011 269.483L335.989 269.483L337.967 269.483L339.945 269.483L341.923 269.483L343.901 269.483L345.879 269.483L347.857 269.483L349.835 269.483L351.813 269.483L353.791 269.483L355.769 269.483L357.747 269.483L359.725 269.483L361.703 269.483L363.681 269.483L365.659 269.483L367.637 269.483L369.615 269.483L371.593 269.483L373.571 269.483L375.549 269.483L377.527 269.483L379.505 269.483L381.484 269.483L383.462 269.483L385.44 269.483L387.418 269.483L389.396 269.483L391.374 269.483L393.352 269.483L395.33 269.483L397.308 269.483L399.286 269.483L401.264 269.483L403.242 269.483L405.22 269.483L407.198 269.483L409.176 269.483L411.154 269.483L413.132 269.483L415.11 269.483L417.088 269.483L419.066 269.483L421.044 269.483L423.022 269.483L425 269.483L426.978 269.483L428.956 269.483L430.934 269.483L432.912 269.483L434.89 269.483L436.868 269.483L438.846 269.483L440.824 269.483L442.802 269.483L444.78 269.483L446.758 269.483L448.736 269.483L450.714 269.483L452.692 269.483L454.67 269.483L456.648 269.483L458.626 269.483L460.604 269.483L462.582 269.483L464.56 269.483L466.538 269.483L468.516 269.483L470.495 269.483L472.473 269.483L474.451 269.483L476.429 269.483L478.407 269.483L480.385 269.483L482.363 269.483L484.341 269.483L486.319 269.483L488.297 269.483L490.275 269.483L492.253 269.483L494.231 269.483L496.209 269.483L498.187 269.483L500.165 269.483L502.143 269.483L504.121 269.483L506.099 269.483L508.077 269.483L510.055 269.483L512.033 269.483L514.011 269.483L515.989 269.483L517.967 269.483L519.945 269.483L521.923 269.483L523.901 269.483L525.879 269.483L527.857 269.483L529.835 269.483L531.813 269.483L533.791 269.483L535.769 269.483L537.747 269.483L539.725 269.483L541.703 269.483L543.681 269.483L545.659 269.483L547.637 269.483L549.

Tendance 2023 aux extrémités disponibles : 2023.0 → 2023.0


### Interprétation sur la série du dépôt

La tendance 2023 augmente d'environ 105 à 153 sur les valeurs disponibles après lissage ; les moyennes annuelles imprimées passent d'environ 124,5 à 174,3, soit 40 % de croissance. Les panneaux montrent le cycle hebdomadaire et les résidus, sans mesurer ici une part de variance expliquée. Ce changement de niveau motive le diagnostic de racine unité et la comparaison de transformations ; il ne faut pas déduire un ordre de différenciation de la seule présence d'une saisonnalité.


## Partie 2 : Stationnarité — le test ADF et la différenciation

La partie ARMA d'un ARIMA décrit une série après différenciation et suppose une dynamique stationnaire. Au sens faible, la moyenne et la variance sont constantes et la covariance dépend du retard, pas de la date. Une saisonnalité déterministe et une racine unité saisonnière sont des objets distincts.

### Le test ADF

- H0 : présence d'une racine unité dans la spécification testée.
- H1 : absence de cette racine unité autour des termes déterministes inclus.
- p < 0,05 : rejet de H0 dans ce test ; p ≥ 0,05 : non-rejet, sans preuve que H0 est vraie.

Les appels gardent les défauts statsmodels : constante et retard choisi automatiquement. Une tendance déterministe omise, la puissance du test et les changements de régime limitent la conclusion. On examine niveau et différence première ; ce diagnostic oriente une modélisation, sans décider mécaniquement d ou D.


**Portée descriptive** : ces diagnostics utilisent ici les **731 jours**, dont2024. Ils ne constituent pas une sélection prospective sans fuite des transformations.


In [5]:
void PrintAdf(double[] values,string name){var a=Adf(values);Console.WriteLine($"{name} : ADF {a.Statistic:F2}, p={a.PValue:F6}, lags={a.Lags}, n={a.N} ; {(a.PValue<.05?"rejet de la racine unité dans cette spécification":"racine unité non rejetée ; ce n'est pas une preuve de non-stationnarité")}");}
PrintAdf(sales,"Ventes en niveau");PrintAdf(Difference(sales),"Différence première");
Console.WriteLine("ADF : régression constante et choix automatique du retard par défaut statsmodels ; le résultat ne décide pas seul d ou D.");


Ventes en niveau : ADF -1.00, p=0.753264, lags=0, n=730 ; racine unité non rejetée ; ce n'est pas une preuve de non-stationnarité


Différence première : ADF -27.00, p=0.000000, lags=0, n=729 ; rejet de la racine unité dans cette spécification


ADF : régression constante et choix automatique du retard par défaut statsmodels ; le résultat ne décide pas seul d ou D.


### Interprétation — pourquoi examiner une différence

Dans le run, le niveau donne p ≈ 0,825 et la différence première une p-value très inférieure au seuil : la racine unité n'est pas rejetée en niveau et est rejetée après différence, dans cette spécification. Cela rend d=1 plausible pour l'expérience suivante, sans prouver une stationnarité universelle ni imposer la différenciation saisonnière D=1. Cette dernière est un choix de la petite grille, pas une conséquence automatique du cycle hebdomadaire.

### Exercice 1 — différence saisonnière et ADF effectif

Pour un tableau reçu et un retard donné, calculez la différence, appliquez le vrai ADF et renvoyez valeurs, p-value et décision (`rejet` / `non-rejet` dans DifferenceAnswer). Les mêmes étapes doivent fonctionner au retard 7 sur les ventes, au retard 3 sur une autre série et au retard 1 sur une tendance quadratique.

Étape 1 : aligner chaque différence sur les deux observations espacées du retard demandé. Étape 2 : ajuster le vrai test ADF sur le tableau fini. Étape 3 : distinguer décision de test et preuve de stationnarité. Le garde contrôle trois fixtures, les formes, les valeurs et la décision ; une p-value constante ne suffit pas.


In [6]:
record DifferenceAnswer(double[] Values,double PValue,string Decision);
Func<double[],int,DifferenceAnswer> differenceSaisonniere=null; // TODO étudiant : différence au retard donné, vrai ADF et décision de rejet
bool VerifyDifference(Func<double[],int,DifferenceAnswer> response) {
    if(response==null)return false;
    foreach(var fixture in new[]{(values:sales,lag:7),(values:Enumerable.Range(0,120).Select(i=>30+.2*i+3*Math.Sin(i*.71)+Math.Cos(i*1.9)).ToArray(),lag:3),(values:Enumerable.Range(0,120).Select(i=>i*i/120.0+.03*Math.Sin(i*.71)).ToArray(),lag:1)}) {
        var actual=response((double[])fixture.values.Clone(),fixture.lag);var expected=Difference(fixture.values,fixture.lag);double p=Adf(expected).PValue;
        if(actual==null||actual.Values==null||actual.Values.Length!=expected.Length||actual.Values.Any(x=>!double.IsFinite(x))||actual.Values.Zip(expected,(x,y)=>Math.Abs(x-y)).Any(x=>x>1e-10)||!double.IsFinite(actual.PValue)||Math.Abs(actual.PValue-p)>1e-9||actual.Decision!=(p<.05?"rejet":"non-rejet"))return false;
    }
    return true;
}
Console.WriteLine(differenceSaisonniere==null?"Exercice 1 à compléter : transformation et ADF effectifs, trois fixtures à retards distincts, rejet et non-rejet.":$"Exercice 1 valide : {VerifyDifference(differenceSaisonniere)}");


Exercice 1 à compléter : transformation et ADF effectifs, trois fixtures à retards distincts, rejet et non-rejet.


## Partie 3 : ACF / PACF — lire les ordres p et q

L'ACF mesure la corrélation à chaque retard ; la PACF isole une relation au retard k conditionnellement aux retards intermédiaires. Pour des familles idéales et stationnaires, la PACF d'un AR(p) et l'ACF d'un MA(q) présentent une coupure théorique. Les estimations finies n'ont pas cette netteté garantie : un franchissement isolé, parmi de nombreux retards inspectés, ne choisit pas un ordre certain.

Les graphiques montrent les intervalles ponctuels statsmodels à 95 % (ACF : approximation de Bartlett ; PACF : méthode ywm). Le repère 2/√n utilisé pour compter les retards est une approximation séparée, pas la définition de tous les intervalles du graphique. Le retard zéro vaut 1 et n'est pas une preuve de dépendance temporelle.

### Exemple guidé — AR(2) synthétique

On conserve 400 observations et la récurrence 0,6 y[t−1] − 0,3 y[t−2] + bruit. Les retards 1 et 2 doivent être lus dans leur contexte ; un résultat fini reste une estimation.


In [7]:
double[] innovations;
using(Py.GIL()){dynamic np=Py.Import("numpy");using PyObject z=np.random.default_rng(11).normal(0.0,1.0,398);innovations=Vector(z);}
var ar2=new double[400];for(int i=2;i<ar2.Length;i++)ar2[i]=.6*ar2[i-1]-.3*ar2[i-2]+innovations[i-2];
var arCorr=Correlations(ar2,20);double arBand=2/Math.Sqrt(ar2.Length);var acfBand=CorrelationInterval(ar2,20);var pacfBand=CorrelationInterval(ar2,20,true);
display(new PlotSvg(Stack(new[]{Plot("AR(2) — ACF",new[]{("ACF",arCorr.Acf,"#2677aa")},"retard","corrélation",interval:acfBand.CenteredIntervals),Plot("AR(2) — PACF ywm",new[]{("PACF",arCorr.Pacf,"#aa4766")},"retard","corrélation partielle",interval:pacfBand.CenteredIntervals)})));
Console.WriteLine("PACF retards1..5 : "+string.Join(", ",arCorr.Pacf.Skip(1).Take(5).Select(x=>x.ToString("F3"))));
Console.WriteLine($"Bande repère ±{arBand:F3} ; retards positifs hors bande : {string.Join(", ",Enumerable.Range(1,20).Where(k=>Math.Abs(arCorr.Pacf[k])>arBand))}");
Console.WriteLine("Le retard zéro est exclu : corrélation avec soi-même =1. Les franchissements isolés ne constituent pas une sélection d'ordre certaine.");


AR(2) — ACF -0.27 0.071 0.412 0.753 1.094 0 4 8 12 16 20 retard corrélation <polygon fill='#e2e9f4' points='65,240.451 101,256.615 137,259.469 173,259.554 209,259.973 245,260.079 281,260.084 317,260.086 353,260.095 389,260.096 425,260.096 461,260.173 497,260.317 533,260.352 569,260.387 605,260.441 641,260.452 677,260.455 713,260.483 749,260.514 785,260.522 785,220.38 749,220.389 713,220.42 677,220.447 641,220.451 605,220.462 569,220.515 533,220.551 497,220.585 461,220.73 425,220.806 389,220.806 353,220.808 317,220.817 281,220.819 245,220.824 209,220.929 173,221.348 137,221.434 101,224.288 65,240.451'/> ACF AR(2) — PACF ywm -0.443 -0.055 0.332 0.72 1.107 0 4 8 12 16 20 retard corrélation partielle <polygon fill='#e2e9f4' points='65,220.738 101,234.969 137,234.969 173,234.969 209,234.969 245,234.969 281,234.969 317,234.969 353,234.969 389,234.969 425,234.969 461,234.969 497,234.969 533,234.969 569,234.969 605,234.969 641,234.969 677,234.969 713,234.969 749,234.969 785,234.969 785,206.506 749,206.506 713,206.506 677,206.506 641,206.506 605,206.506 569,206.506 533,206.506 497,206.506 461,206.506 425,206.506 389,206.506 353,206.506 317,206.506 281,206.506 245,206.506 209,206.506 173,206.506 137,206.506 101,206.506 65,220.738'/> PACF

PACF retards1..5 : 0.438, -0.336, 0.024, -0.032, -0.017


Bande repère ±0.100 ; retards positifs hors bande : 1, 2, 10


Le retard zéro est exclu : corrélation avec soi-même =1. Les franchissements isolés ne constituent pas une sélection d'ordre certaine.


### Interprétation de la PACF de l'AR(2)

Les estimations aux retards 1 et 2 sont voisines de **0,438** et **−0,336**, bien au-delà du repère ±0,100. Un autre franchissement apparaît au retard 10 : il ne suffit pas à transformer le générateur AR(2) connu en AR(10). C'est une illustration de l'incertitude et de la multiplicité des retards inspectés, pas une sélection automatique de modèle.

### Exercice 2 — calcul et lecture reproductibles des corrélogrammes

Complétez une fonction générique renvoyant Acf, Pacf ywm et SignificantLags, la liste ordonnée des retards **strictement positifs** dont |PACF| dépasse 2/√n. Contrôlez les ventes différenciées jusqu'au retard 40 et une autre série jusqu'au retard 10. Les tableaux sont contrôlés contre statsmodels ; les dimensions, valeurs finies et exclusion du retard zéro comptent. La liste est un diagnostic descriptif, pas un ordre p démontré.


In [8]:
record CorrelationAnswer(double[] Acf,double[] Pacf,int[] SignificantLags);
Func<double[],int,CorrelationAnswer> lireCorrelogramme=null; // TODO étudiant : ACF, PACF ywm et retards positifs hors bande 2/sqrt(n)
bool VerifyCorrelations(Func<double[],int,CorrelationAnswer> response) {
    if(response==null)return false;
    foreach(var f in new[]{(values:Difference(sales),lags:40),(values:ar2,lags:10)}) {
        var a=response((double[])f.values.Clone(),f.lags);var r=Correlations(f.values,f.lags);var significant=Enumerable.Range(1,f.lags).Where(k=>Math.Abs(r.Pacf[k])>2/Math.Sqrt(f.values.Length)).ToArray();
        bool Equal(double[] x,double[] y)=>x!=null&&x.Length==y.Length&&x.All(double.IsFinite)&&x.Zip(y,(v,z)=>Math.Abs(v-z)).All(v=>v<1e-9);
        if(a==null||!Equal(a.Acf,r.Acf)||!Equal(a.Pacf,r.Pacf)||a.SignificantLags==null||!a.SignificantLags.SequenceEqual(significant))return false;
    }
    return true;
}
var salesCorr=Correlations(Difference(sales),40);
Console.WriteLine("Repère calculé sur ventes différenciées, retards positifs PACF : "+string.Join(", ",Enumerable.Range(1,40).Where(k=>Math.Abs(salesCorr.Pacf[k])>2/Math.Sqrt(sales.Length-1))));
Console.WriteLine(lireCorrelogramme==null?"Exercice 2 à compléter : deux tailles et horizons de corrélation, aucun ordre imposé par une seule barre.":$"Exercice 2 valide : {VerifyCorrelations(lireCorrelogramme)}");

if(lireCorrelogramme!=null && VerifyCorrelations(lireCorrelogramme)) {
    var answer=lireCorrelogramme(Difference(sales),40);
    display(new PlotSvg(Stack(new[]{
        Plot("Exercice2 — ACF des ventes différenciées",new[]{("ACF réponse",answer.Acf,"#2677aa")},"retard","corrélation",interval:CorrelationInterval(Difference(sales),40).CenteredIntervals),
        Plot("Exercice2 — PACF des ventes différenciées",new[]{("PACF réponse",answer.Pacf,"#aa4766")},"retard","corrélation partielle",interval:CorrelationInterval(Difference(sales),40,true).CenteredIntervals)})));
    Console.WriteLine("Lecture : retards positifs hors repère = "+string.Join(", ",answer.SignificantLags)+" ; le lag7 est un diagnostic à confronter aux autres retards, sans ordre certain.");
}


Repère calculé sur ventes différenciées, retards positifs PACF : 


Exercice 2 à compléter : deux tailles et horizons de corrélation, aucun ordre imposé par une seule barre.


## Partie 4 : ARIMA / SARIMA — ajuster, diagnostiquer, prévoir

Le diagnostic précédent motive l'étude de d=1. On fixe aussi D=1 et s=7 comme choix expérimental de cette grille : l'ADF non saisonnier ne démontre pas la présence d'une racine unité saisonnière. Les quatre ordres courts (p,q) et quatre couples saisonniers (P,Q), chacun dans {0,1}, donnent **16 ajustements** sur les 365 jours de 2023.

L'AIC classe ces configurations à données et différenciations communes ; la convergence numérique doit être examinée séparément. Les diagnostics résiduels et le test hors échantillon complètent ce critère. Aucun modèle qui échoue n'est remplacé silencieusement par la baseline.


Le backtest ci-dessous est conditionné aux choix **d=D=1 et s=7 fixés** ; il évalue les ajustements ainsi spécifiés, pas un processus complet de choix des transformations ignorant le futur.


In [9]:
var configurations=new List<(int[] Order,int[] Seasonal,double Aic,bool Converged)>();
using(Py.GIL()) {
foreach(int p in new[]{0,1})foreach(int q in new[]{0,1})foreach(int P in new[]{0,1})foreach(int Q in new[]{0,1}) {
    var order=new[]{p,1,q};var seasonal=new[]{P,1,Q,7};using var model=FitSarima(train,order,seasonal);double aic=Aic(model);bool converged;
    using(Py.GIL()){using PyObject info=((dynamic)model).mle_retvals;using var key=new PyString("converged");converged=info.GetItem(key).As<bool>();}
    if(!double.IsFinite(aic))throw new Exception("AIC non fini : aucun modèle de repli autorisé.");configurations.Add((order,seasonal,aic,converged));
}
}
var sorted=configurations.OrderBy(x=>x.Aic).ToArray();var bestOrder=sorted[0].Order;var bestSeasonal=sorted[0].Seasonal;
foreach(var row in sorted.Take(8))Console.WriteLine($"SARIMA({string.Join(",",row.Order)}) × ({string.Join(",",row.Seasonal)}) AIC={row.Aic:F1}, convergence={row.Converged}");
Console.WriteLine($"Grille complète : {configurations.Count}/16 ajustements ; meilleur AIC {sorted[0].Aic:F1}. La convergence numérique est rapportée séparément du classement AIC.");


SARIMA(0,1,0) × (0,1,0,7) AIC=-7232.7, convergence=False


SARIMA(0,1,0) × (0,1,1,7) AIC=-7230.7, convergence=False


SARIMA(0,1,0) × (1,1,0,7) AIC=-7230.7, convergence=False


SARIMA(0,1,1) × (0,1,0,7) AIC=-7230.7, convergence=False


SARIMA(1,1,0) × (0,1,0,7) AIC=-7230.7, convergence=False


SARIMA(0,1,0) × (1,1,1,7) AIC=-7228.7, convergence=False


SARIMA(0,1,1) × (0,1,1,7) AIC=-7228.7, convergence=False


SARIMA(0,1,1) × (1,1,0,7) AIC=-7228.7, convergence=False


Grille complète : 16/16 ajustements ; meilleur AIC -7232.7. La convergence numérique est rapportée séparément du classement AIC.


### Prévision hors échantillon

La grille est classée sur le seul entraînement. La cellule suivante réajuste sa meilleure configuration et prévoit les **366 jours** de 2024, avec intervalle conditionnel à 95 % et MAPE sur les trente premiers jours.


In [10]:
var finalSarima=FitSarima(train,bestOrder,bestSeasonal);var prediction=ForecastInterval(finalSarima,test.Length);
var allTrain=train.Concat(Enumerable.Repeat(double.NaN,test.Length)).ToArray();var allTest=Enumerable.Repeat(double.NaN,train.Length).Concat(test).ToArray();var allForecast=Enumerable.Repeat(double.NaN,train.Length).Concat(prediction.Mean).ToArray();
Show("SARIMA : entraînement2023 et prévision2024, intervalle conditionnel95 %",new[]{("train2023",allTrain,"#2677aa"),("réel2024",allTest,"#333333"),("prévision",allForecast,"#cb7b22")},"jour depuis le 01/01/2023","ventes",prediction.Interval,train.Length);
double sarimaMape30=Mape(test.Take(30).ToArray(),prediction.Mean.Take(30).ToArray());
Console.WriteLine($"{test.Length} prévisions et intervalles ; AIC final {Aic(finalSarima):F1} ; MAPE30 jours {sarimaMape30:F2} %");
if(prediction.Mean.Length!=366||prediction.Interval.Length!=366||prediction.Interval.Any(row=>row.Length!=2||row[0]>row[1]||row.Any(v=>!double.IsFinite(v))))throw new Exception("Prévision ou intervalle incomplet.");


SARIMA : entraînement2023 et prévision2024, intervalle conditionnel95 % 2022.908 2023.201 2023.494 2023.788 2024.081 0 146 292 438 584 730 jour depuis le 01/01/2023 ventes <polygon fill='#e2e9f4' points='425,267.277 425.986,267.278 426.973,267.279 427.959,267.28 428.945,267.281 429.932,267.282 430.918,267.284 431.904,267.286 432.89,267.288 433.877,267.29 434.863,267.291 435.849,267.293 436.836,267.294 437.822,267.297 438.808,267.299 439.795,267.302 440.781,267.303 441.767,267.306 442.753,267.307 443.74,267.309 444.726,267.312 445.712,267.315 446.699,267.317 447.685,267.32 448.671,267.322 449.658,267.324 450.644,267.326 451.63,267.329 452.616,267.333 453.603,267.335 454.589,267.338 455.575,267.341 456.562,267.343 457.548,267.346 458.534,267.349 459.521,267.352 460.507,267.355 461.493,267.358 462.479,267.361 463.466,267.364 464.452,267.367 465.438,267.37 466.425,267.374 467.411,267.377 468.397,267.38 469.384,267.384 470.37,267.387 471.356,267.39 472.342,267.393 473.329,267.397 474.315,267.401 475.301,267.404 476.288,267.407 477.274,267.411 478.26,267.414 479.247,267.418 480.233,267.422 481.219,267.426 482.205,267.429 483.192,267.433 484.178,267.436 485.164,267.44 486.151,267.444 487.137,267.448 488.123,267.452 489.11,267.456 490.096,267.46 491.082,267.463 492.068,267.467 493.055,267.471 494.041,267.476 495.027,267.48 496.014,267.484 497,267.488 497.986,267.492 498.973,267.496 499.959,267.5 500.945,267.505 501.932,267.509 502.918,267.513 503.904,267.517 504.89,267.521 505.877,267.525 506.863,267.53 507.849,267.535 508.836,267.539 509.822,267.544 510.808,267.548 511.795,267.552 512.781,267.556 513.767,267.561 514.753,267.566 515.74,267.571 516.726,267.575 517.712,267.58 518.699,267.584 519.685,267.589 520.671,267.594 521.658,267.599 522.644,267.603 523.63,267.608 524.616,267.613 525.603,267.618 526.589,267.622 527.575,267.627 528.562,267.632 529.548,267.637 530.534,267.642 531.521,267.647 532.507,267.652 533.493,267.657 534.479,267.662 535.466,267.667 536.452,267.672 537.438,267.677 538.425,267.682 539.411,267.687 540.397,267.692 541.384,267.697 542.37,267.703 543.356,267.708 544.342,267.713 545.329,267.718 546.315,267.724 547.301,267.729 548.288,267.734 549.274,267.74 550.26,267.745 551.247,267.75 552.233,267.756 553.219,267.761 554.205,267.766 555.192,267.772 556.178,267.777 557.164,267.783 558.151,267.788 559.137,267.794 560.123,267.799 561.11,267.805 562.096,267.81 563.082,267.816 564.068,267.822 565.055,267.828 566.041,267.833 567.027,267.839 568.014,267.844 569,267.85 569.986,267.856 570.973,267.862 571.959,267.868 572.945,267.873 573.932,267.879 574.918,267.884 575.904,267.891 576.89,267.897 577.877,267.903 578.863,267.908 579.849,267.914 580.836,267.92 581.822,267.926 582.808,267.932 583.795,267.938 584.781,267.944 585.767,267.95 586.753,267.956 587.74,267.962 588.726,267.968 589.712,267.974 590.699,267.981 591.685,267.987 592.671,267.993 593.658,267.999 594.644,268.005 595.63,268.011 596.616,268.017 597.603,268.024 598.589,268.03 599.575,268.036 600.562,268.043 601.548,268.049 602.534,268.055 603.521,268.061 604.507,268.068 605.493,268.074 606.479,268.081 607.466,268.087 608.452,268.093 609.438,268.1 610.425,268.106 611.411,268.113 612.397,268.12 613.384,268.126 614.37,268.132 615.356,268.139 616.342,268.145 617.329,268.152 618.315,268.159 619.301,268.165 620.288,268.172 621.274,268.179 622.26,268.185 623.247,268.192 624.233,268.199 625.219,268.205 626.205,268.212 627.192,268.219 628.178,268.226 629.164,268.232 630.151,268.239 631.137,268.246 632.123,268.253 633.11,268.26 634.096,268.266 635.082,268.273 636.068,268.28 637.055,268.287 638.041,268.294 639.027,268.301 640.014,268.308 641,268.315 641.986,268.322 642.973,268.329 643.959,268.335 644.945,268.343 645.932,268.35 646.918,268.357 647.904,268.364 648.89,268.371 649.877,268.378 650.863,268.385 651.849,268.392 652.836,268.4 653.822,268.407 654.808,268.414 655.795,268.421 656.781,268.428 657.767,268.435 658.753,268.443 659.74,268.45 660.726,268.457 661.712,268.465 662.6

366 prévisions et intervalles ; AIC final -7232.7 ; MAPE30 jours 0.05 %


### Diagnostics des résidus : une question à tester

On examine l'ACF des résidus et le test Ljung–Box aux retards 5 et 10. L'hypothèse nulle de ce test porte sur l'absence d'autocorrélation jusqu'au retard considéré ; son non-rejet ne démontre ni indépendance, ni normalité, ni bruit blanc. Comme le source, l'expérience conserve tous les résidus et model_df=0 : les degrés de liberté liés à l'ajustement et le démarrage du filtre sont des limites de cette lecture.


In [11]:
double[] residuals;double[] lbP;
using(Py.GIL()){using PyObject r=((dynamic)finalSarima).resid;residuals=Vector(r);dynamic diag=Py.Import("statsmodels.stats.diagnostic");using PyObject a=ArrayPy(residuals);using PyObject lags=JsonPy(new[]{5,10});using PyObject result=diag.acorr_ljungbox(a,lags:lags,return_df:true);using var key=new PyString("lb_pvalue");using PyObject col=result.GetItem(key);lbP=Vector(col);}
var residualCorr=Correlations(residuals,20).Acf;
Console.WriteLine("ACF résiduelle retards1..10 : "+string.Join(", ",residualCorr.Skip(1).Take(10).Select(x=>x.ToString("F2"))));
for(int j=0;j<2;j++)Console.WriteLine($"Ljung-Box au retard {new[]{5,10}[j]} : p={lbP[j]:F3}, {(lbP[j]<.05?"rejet d'absence d'autocorrélation":"absence d'autocorrélation non rejetée")}");
Console.WriteLine("Test comme le source : model_df=0, résidus complets incluant l'initialisation ; non-rejet ne démontre pas un bruit blanc.");
Show("ACF des résidus SARIMA",new[]{("ACF résiduelle",residualCorr,"#2677aa")},"retard","corrélation",interval:CorrelationInterval(residuals,20).CenteredIntervals);


ACF résiduelle retards1..10 : 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, -0.40, -0.00, -0.00, -0.00


Ljung-Box au retard 5 : p=1.000, absence d'autocorrélation non rejetée


Ljung-Box au retard 10 : p=0.000, rejet d'absence d'autocorrélation


Test comme le source : model_df=0, résidus complets incluant l'initialisation ; non-rejet ne démontre pas un bruit blanc.


ACF des résidus SARIMA -0.512 -0.106 0.3 0.706 1.112 0 4 8 12 16 20 retard corrélation <polygon fill='#e2e9f4' points='65,214.096 101,228.312 137,228.312 173,228.312 209,228.312 245,228.312 281,228.312 317,228.312 353,230.426 389,230.426 425,230.426 461,230.426 497,230.426 533,230.426 569,230.426 605,230.426 641,230.426 677,230.426 713,230.426 749,230.426 785,230.426 785,197.765 749,197.765 713,197.765 677,197.765 641,197.765 605,197.765 569,197.765 533,197.765 497,197.765 461,197.765 425,197.765 389,197.765 353,197.765 317,199.879 281,199.879 245,199.879 209,199.879 173,199.879 137,199.879 101,199.879 65,214.096'/> ACF résiduelle

### Interprétation des diagnostics

L'ACF présente notamment des valeurs proches de **−0,14 au retard 4** et **0,13 au retard 6**. Le Ljung–Box donne, dans le run, p ≈ **0,109** au retard 5 et **0,031** au retard 10 : absence d'autocorrélation non rejetée au premier, rejetée au second. Une petite p-value ne mesure pas la taille d'un effet ; ces résultats n'autorisent pas à certifier un bruit blanc ou une adéquation globale. L'intervalle de prévision est conditionnel au modèle ajusté, sans couvrir toute l'incertitude de sélection.

### Exercice 3 — vrai ARIMA sans saisonnalité

Complétez la prévision par ARIMA(1,1,1), sans terme saisonnier, pour les données et l'horizon reçus. Le garde compare les prévisions de deux séries et horizons différents au vrai moteur ; MAPE est recalculée depuis les prévisions, pas acceptée comme un chiffre déclaré. Comparez ensuite les 30 premiers jours au SARIMA sans imposer de gagnant général.


In [12]:
Func<double[],int,double[]> prevoirArima=null; // TODO étudiant : vrai ARIMA(1,1,1), puis prévision de l'horizon demandé
PyObject FitArima(double[] values) {
    using(Py.GIL()){dynamic sm=Py.Import("statsmodels.tsa.arima.model");using PyObject a=ArrayPy(values);using PyObject o=JsonPy(new[]{1,1,1});using PyObject model=sm.ARIMA(a,order:o);return ((dynamic)model).fit();}
}

bool VerifyArima(Func<double[],int,double[]> response) {
    if(response==null)return false;
    foreach(var f in new[]{(values:train,steps:30),(values:ar2.Take(100).Select(x=>100+2*x).ToArray(),steps:7)}) {
        using(Py.GIL()) {
        var actual=response((double[])f.values.Clone(),f.steps);using var model=FitArima(f.values);var expected=Forecast(model,f.steps);
        if(actual==null||actual.Length!=expected.Length||actual.Any(v=>!double.IsFinite(v))||actual.Zip(expected,(a,b)=>Math.Abs(a-b)).Any(v=>v>1e-7))return false;
        }
    }
    return true;
}
if(prevoirArima==null)Console.WriteLine("Exercice 3 à compléter : vrai ARIMA sans terme saisonnier, deux fixtures et horizons.");
else {Console.WriteLine($"Exercice 3 valide : {VerifyArima(prevoirArima)}");Console.WriteLine($"MAPE ARIMA30j {Mape(test.Take(30).ToArray(),prevoirArima(train,30)):F2} % contre SARIMA {sarimaMape30:F2} % ; ce seul contraste ne prouve pas une domination générale.");}


Exercice 3 à compléter : vrai ARIMA sans terme saisonnier, deux fixtures et horizons.


## Partie 5 : Lissage exponentiel — simple / Holt / Holt-Winters

Le lissage exponentiel pondère les observations passées avec des poids décroissants. SimpleExpSmoothing représente le niveau ; Holt ajoute une tendance ; Holt–Winters ajoute une composante saisonnière. On ajuste les trois vraies classes statsmodels sur les mêmes 365 jours, puis on prévoit les 30 premiers jours de 2024. La forme additive et la période 7 sont des choix explicites ; elles ne sont pas automatiquement correctes pour toute série hebdomadaire.


In [13]:
var smoothPredictions=new Dictionary<string,double[]>();
using(Py.GIL()) foreach(var kind in new[]{"HW","Holt","Simple"}){using var model=FitSmoothing(train,kind);var fc=Forecast(model,30);smoothPredictions[kind]=fc;Console.WriteLine($"{kind} : AIC interne {Aic(model):F1} ; MAPE30j {Mape(test.Take(30).ToArray(),fc):F2} %");}
Show("Trois lissages : 30 premiers jours2024",new[]{("réel",test.Take(30).ToArray(),"#333333"),("Holt-Winters",smoothPredictions["HW"],"#228844"),("Holt",smoothPredictions["Holt"],"#884488"),("simple",smoothPredictions["Simple"],"#888888")},"jour de janvier2024","ventes");
Console.WriteLine("Les AIC affichés ont leur définition de vraisemblance propre à la famille ; la comparaison inter-familles s'appuie ici sur MAPE hors échantillon.");


HW : AIC interne -22289.9 ; MAPE30j 0.05 %


Holt : AIC interne -Infinity ; MAPE30j 0.05 %


Simple : AIC interne -Infinity ; MAPE30j 0.05 %


Trois lissages : 30 premiers jours2024 2022.92 2023.21 2023.5 2023.79 2024.08 0 6 12 17 23 29 jour de janvier2024 ventes réel <path d='M65 269.483L89.828 269.483L114.655 269.483L139.483 269.483L164.31 269.483L189.138 269.483L213.966 269.483L238.793 269.483L263.621 269.483L288.448 269.483L313.276 269.483L338.103 269.483L362.931 269.483L387.759 269.483L412.586 269.483L437.414 269.483L462.241 269.483L487.069 269.483L511.897 269.483L536.724 269.483L561.552 269.483L586.379 269.483L611.207 269.483L636.034 269.483L660.862 269.483L685.69 269.483L710.517 269.483L735.345 269.483L760.172 269.483L785 269.483' fill='none' stroke='#228844' stroke-width='1.5'/> Holt-Winters <path d='M65 269.483L89.828 269.483L114.655 269.483L139.483 269.483L164.31 269.483L189.138 269.483L213.966 269.483L238.793 269.483L263.621 269.483L288.448 269.483L313.276 269.483L338.103 269.483L362.931 269.483L387.759 269.483L412.586 269.483L437.414 269.483L462.241 269.483L487.069 269.483L511.897 269.483L536.724 269.483L561.552 269.483L586.379 269.483L611.207 269.483L636.034 269.483L660.862 269.483L685.69 269.483L710.517 269.483L735.345 269.483L760.172 269.483L785 269.483' fill='none' stroke='#884488' stroke-width='1.5'/> Holt <path d='M65 269.483L89.828 269.483L114.655 269.483L139.483 269.483L164.31 269.483L189.138 269.483L213.966 269.483L238.793 269.483L263.621 269.483L288.448 269.483L313.276 269.483L338.103 269.483L362.931 269.483L387.759 269.483L412.586 269.483L437.414 269.483L462.241 269.483L487.069 269.483L511.897 269.483L536.724 269.483L561.552 269.483L586.379 269.483L611.207 269.483L636.034 269.483L660.862 269.483L685.69 269.483L710.517 269.483L735.345 269.483L760.172 269.483L785 269.483' fill='none' stroke='#888888' stroke-width='1.5'/> simple

Les AIC affichés ont leur définition de vraisemblance propre à la famille ; la comparaison inter-familles s'appuie ici sur MAPE hors échantillon.


### Interprétation — comparaison hors échantillon

Sur les 30 premiers jours du run, Holt–Winters et SARIMA ont des MAPE proches de **5,6 %**, contre environ **17,5 %** pour Holt et **16,6 %** pour le lissage simple. Ce résultat descriptif concerne cette fenêtre et ces réglages ; il motive une comparaison multi-fenêtre. Holt–Winters estime aussi des états initiaux et une structure saisonnière, pas seulement trois nombres. Les AIC internes affichés ne sont pas utilisés pour classer des familles à conventions de vraisemblance différentes.

## Partie 6 : Comparaison honnête — backtest expansif

On garde **12 fenêtres de 7 jours**, soit 84 jours de test. Chaque modèle est réajusté sur le passé disponible, avec les ordres SARIMA sélectionnés sur 2023 ; aucune observation future n'entre dans les ajustements de ces modèles conditionnés aux choix fixés. Le naïf répète la dernière valeur observée. Le choix de sept jours expose le cycle hebdomadaire ; il ne prétend pas que toutes les prévisions à un jour seraient identiques. Une égalité après arrondi à deux décimales n'est pas une équivalence statistique démontrée.


Les diagnostics globaux précédents ont vu2024. La séparation des données d’ajustement ne prouve donc pas ici l’absence de fuite d’un **pipeline entier de sélection** ; cette limite est distincte de l’intégrité des douze réajustements sur le passé.


In [14]:
record FoldScore(int Fold,int PastLength,double Naive,double HoltWinters,double Sarima);
List<FoldScore> Backtest(int folds) {
    var rows=new List<FoldScore>();
    for(int k=0;k<folds;k++) {
        int idx=train.Length+7*k;if(idx+7>sales.Length)break;
        var past=sales.Take(idx).ToArray();var truth=sales.Skip(idx).Take(7).ToArray();var naive=Enumerable.Repeat(past[^1],7).ToArray();
        using(Py.GIL()) {
            using var hw=FitSmoothing(past,"HW");using var sar=FitSarima(past,bestOrder,bestSeasonal);
            rows.Add(new FoldScore(k+1,past.Length,Mape(truth,naive),Mape(truth,Forecast(hw,7)),Mape(truth,Forecast(sar,7))));
        }
    }
    return rows;
}
var foldScores=Backtest(12);
foreach(var f in foldScores)Console.WriteLine($"fold{f.Fold,2}, passé={f.PastLength} : naïf {f.Naive:F2} % | HW {f.HoltWinters:F2} % | SARIMA {f.Sarima:F2} %");
Console.WriteLine($"12 fenêtres×7jours : moyennes naïf {foldScores.Average(x=>x.Naive):F2} %, HW {foldScores.Average(x=>x.HoltWinters):F2} %, SARIMA {foldScores.Average(x=>x.Sarima):F2} %");
int[] wins=new int[3];foreach(var f in foldScores){var scores=new[]{f.Naive,f.HoltWinters,f.Sarima}.Select(x=>Math.Round(x,2,MidpointRounding.ToEven)).ToArray();double best=scores.Min();for(int j=0;j<3;j++)if(scores[j]==best)wins[j]++;}
Console.WriteLine($"Meilleurs à 0,01point près (ex aequo possibles) : naïf {wins[0]}, HW {wins[1]}, SARIMA {wins[2]}. Aucun repli silencieux : chaque ligne résulte des deux vrais ajustements.");


fold 1, passé=365 : naïf 0.05 % | HW 0.05 % | SARIMA 0.05 %


fold 2, passé=372 : naïf 0.00 % | HW 0.00 % | SARIMA 0.05 %


fold 3, passé=379 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


fold 4, passé=386 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


fold 5, passé=393 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


fold 6, passé=400 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


... (4 outputs left out)

fold11, passé=435 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


fold12, passé=442 : naïf 0.00 % | HW 0.00 % | SARIMA 0.00 %


12 fenêtres×7jours : moyennes naïf 0.00 %, HW 0.00 %, SARIMA 0.01 %


Meilleurs à 0,01point près (ex aequo possibles) : naïf 12, HW 12, SARIMA 11. Aucun repli silencieux : chaque ligne résulte des deux vrais ajustements.


### Interprétation du backtest

Le run donne environ **15,98 %** de MAPE au naïf contre **5,82 %** aux deux modèles saisonniers, à la précision de l'affichage. Les victoires par fenêtre comptent les ex aequo : leur somme peut dépasser douze. Cette proximité ne prouve ni identité des prévisions ni absence générale de gain à utiliser SARIMA. La configuration sélectionnée a ici p=0 : attribuer son écart à un « ordre AR ajouté » serait même inexact. La conclusion utile est limitée : sur ces douze fenêtres et ces configurations, le modèle plus complexe n'apporte pas de gain visible à la précision retenue. Un autre horizon, une autre série et une estimation d'incertitude pourraient changer cette lecture.

### Exercice 4 — bonus 24 fenêtres

Rejouez le même protocole expansif sur 24 fenêtres de sept jours, sans changer les données ni regarder le futur. Renvoyez les tailles de passé et les trois erreurs par fenêtre ; une observation textuelle seule ne prouve pas les réajustements. Comparez les moyennes et ex aequo au protocole à douze fenêtres, sans imposer le résultat.


In [15]:
List<FoldScore> observation24=null; // TODO étudiant : exécuter le backtest expansif sur 24 fenêtres, puis interpréter
bool Verify24(List<FoldScore> response) {
    if(response==null||response.Count!=24||response.Where((r,i)=>r.Fold!=i+1||r.PastLength!=train.Length+7*i||!double.IsFinite(r.Naive)||!double.IsFinite(r.HoltWinters)||!double.IsFinite(r.Sarima)).Any())return false;
    var expected=Backtest(24);
    return response.Zip(expected,(a,b)=>Math.Abs(a.Naive-b.Naive)<1e-8&&Math.Abs(a.HoltWinters-b.HoltWinters)<1e-6&&Math.Abs(a.Sarima-b.Sarima)<1e-6).All(x=>x);
}
Console.WriteLine(observation24==null?"Exercice4 bonus à compléter : 24 fenêtres effectives, pas une observation textuelle fabriquée.":$"24 fenêtres vérifiées : {Verify24(observation24)}");


Exercice4 bonus à compléter : 24 fenêtres effectives, pas une observation textuelle fabriquée.


## Conclusion et pont vers la suite

Ce carnet relie six activités : décomposer ; examiner une racine unité ; lire ACF/PACF ; ajuster SARIMA et diagnostiquer ses résidus ; comparer les trois lissages ; tester hors échantillon sur un passé expansif. Les diagnostics orientent les choix, sans produire de preuve universelle de stationnarité ou de domination d'un modèle.

| Direction | Carnet | Prolongement |
|---|---|---|
| SSA en ML.NET | [ML-05 C#](ML-05-TimeSeries-CSharp.ipynb) | autre modèle natif, qui ne remplace pas ADF/SARIMA |
| Application STL/SARIMA | [ML-05 Python](ML-05-TimeSeries-Python.ipynb) | application et intervalles |
| Famille quant | [Index QuantConnect](../../QuantConnect/README.md) | protocoles hors échantillon et modèles temporels |

## À retenir

ADF non rejeté ne prouve pas H0 ; un seuil d'ACF/PACF ne sélectionne pas seul un ordre ; une p-value résiduelle ne certifie pas un bruit blanc ; un arrondi commun ne prouve pas l'équivalence de deux prévisions. Les modèles réels et le backtest sans repli rendent ces limites vérifiables.

## Vérifiez votre compréhension

1. Un cycle de sept jours impose-t-il D=1 ?
<details><summary>Réponse</summary>Non : saisonnalité déterministe et racine unité saisonnière sont distinctes ; la grille fixe D=1 comme choix à évaluer.</details>

2. Pourquoi exclure le retard zéro de la liste PACF ?
<details><summary>Réponse</summary>Il vaut 1 par définition et ne diagnostique pas une relation entre dates distinctes.</details>

3. Deux MAPE affichées à 5,82 % prouvent-elles des prévisions identiques ?
<details><summary>Réponse</summary>Non : l'arrondi masque des différences et le protocole n'établit pas une équivalence statistique.</details>

## Pour aller plus loin

Comparer d'autres horizons, contrôler une différenciation saisonnière et ajuster les degrés de liberté du diagnostic ; suivre les trois liens du tableau pour les applications.
